In [ ]:
%%capture
%pip install ultralytics roboflow

In [ ]:
projeto_path = "wcama2026-melipona-tracking"
training_name = "yolo11n"

dataset_path = "dataset"

In [ ]:
from ultralytics import YOLO

Fine_Tuning = True # Takes the best result and trains it a bit more.
Resume = False # If False, it starts a new training instead of resuming from where the last one stopped.

if Fine_Tuning: # Takes the best result and trains it a bit more.
  model = YOLO(f'{projeto_path}/{training_name}/weights/best.pt')

  results = model.train(
    data=f'{dataset_path}/data.yaml',
    cfg="configs/hyp.yaml", # File containing the hyperparameters used in training.
    epochs=60,     # How many NEW epochs you want to run.
    batch=96,      # Forces a large batch size (the T4 GPU can handle the Nano model).
    workers=6,     # Uses all CPU cores to prepare the images quickly.
    cache='disk',  # Loads images into cache. This speeds things up A LOT.
    imgsz=640,
    project=projeto_path,
    name=training_name # Name of the experiment folder.
  )
else:
  if Resume: # Resume in case of interruption.
    # Loads "last.pt" (last saved state before crashing).
    model = YOLO(f'{projeto_path}/{training_name}/weights/last.pt')
    # Recovers exactly where it left off (optimizer, learning rate, epoch number) and finishes the remaining epochs.
    results = model.train(resume=True)

  else: # Start a new training from scratch.
    # Load a model
    model = YOLO('yolo11n.pt')  # load a pretrained model (recommended for training)

    results = model.train(
      data=f'{dataset_path}/data.yaml',
      cfg="configs/hyp.yaml", # File containing the hyperparameters used in training.
      epochs=90,      # Approximately 3h:30min.
      batch=96,       # Forces a large batch size (the T4 GPU can handle the Nano model).
      workers=6,      # Uses all CPU cores to prepare the images quickly.
      cache='disk',   # Loads images into cache. This speeds things up A LOT.
      imgsz=640,
      project=projeto_path,
      name=training_name # Name of the experiment folder.
    )

Performs the model testing with the test images.

In [ ]:
"""
This script loads a trained YOLO model and evaluates its performance on a test dataset.
It runs validation using the 'test' split specified in the data.yaml configuration, 
generates performance plots, and prints both general and per-class evaluation metrics 
(such as mAP50-95, mAP50, mAP75, Precision, and Recall) to the console.
"""
from ultralytics import YOLO
import os

# --- 1. Load the trained model ---
# Make sure the path is correct!
MODEL_PATH = f'{projeto_path}/{training_name}/weights/best.pt'

# Check if the model file exists before loading
if not os.path.exists(MODEL_PATH):
    print(f"Error: The model file was not found at: {MODEL_PATH}")
    print("Check if the training was completed successfully and if the path is correct.")
else:
    # Load your best model
    model = YOLO(MODEL_PATH)
    print("Model loaded successfully!")

    # --- 2. Run validation on the TEST set ---
    # The data.yaml file contains the paths for the train, validation, and test sets.
    # We use split='test' to force evaluation on the test set.
    # plots=True will generate and save the metric plots.
    print("Starting validation on the test set...")
    metrics = model.val(
        data=f'{dataset_path}/data.yaml',
        split='test',
        imgsz=640,
        plots=True,
        project=projeto_path, # Optional: to organize the results
        name=f'{training_name}_inference' # Optional: folder name for this evaluation
    )
    print("Validation completed!")

    # --- 3. Access and print the metrics ---
    print("\n--- General Metrics ---")
    print(f"mAP50-95 (general): {metrics.box.map:.4f}")
    print(f"mAP50 (general):    {metrics.box.map50:.4f}")
    print(f"mAP75 (general):    {metrics.box.map75:.4f}")

    print("\n--- Performance Metrics on the Test Set (per class) ---")

    names = metrics.names  # {id: name}
    map50_95 = metrics.box.maps
    map50 = metrics.box.all_ap[:, 0]
    map75 = metrics.box.all_ap[:, 5]

    precision = metrics.box.p
    recall = metrics.box.r

    for class_id, class_name in names.items():
        idx = int(class_id)

        print(f"\nClass: {class_name} (id={idx})")
        print(f"  mAP50-95: {map50_95[idx]:.4f}")
        print(f"  mAP50:    {map50[idx]:.4f}")
        print(f"  mAP75:    {map75[idx]:.4f}")
        print(f"  Precision:{precision[idx]:.4f}")
        print(f"  Recall:   {recall[idx]:.4f}")


In [ ]:
"""
This script evaluates a trained YOLO model by randomly selecting a subset of images
from a test directory and running inference on them. The predictions (images with 
drawn bounding boxes) are then saved to a designated output folder.
"""
from ultralytics import YOLO
import os
import glob # Library to find files matching a specified pattern
import random # Import the random library

# --- 1. Define paths ---
MODEL_PATH = f'{projeto_path}/{training_name}/weights/best.pt'
TEST_IMAGES_DIR = f'{dataset_path}/test/images'
NUM_IMAGES_TO_SELECT = 80 # Define the desired number of images

# --- 2. Find all images and select a random sample ---
# Use a pattern to find all common image types
padrao_imagens = os.path.join(TEST_IMAGES_DIR, '*.jpg') # You can add more formats if needed
lista_completa_imagens = glob.glob(padrao_imagens)

# Check if any images were found
if not lista_completa_imagens:
    print(f"No images found in the folder: {TEST_IMAGES_DIR}")
else:
    print(f"Total images found: {len(lista_completa_imagens)}")

    # Ensure we don't try to select more images than available
    if len(lista_completa_imagens) < NUM_IMAGES_TO_SELECT:
        print(f"Warning: The number of images found ({len(lista_completa_imagens)}) is less than requested ({NUM_IMAGES_TO_SELECT}).")
        print("Using all available images.")
        amostra_de_imagens = lista_completa_imagens
    else:
        # Select random images from the complete list
        amostra_de_imagens = random.sample(lista_completa_imagens, NUM_IMAGES_TO_SELECT)

    print(f"Processing a random sample of {len(amostra_de_imagens)} images...")

    # --- 3. Load the model ---
    model = YOLO(MODEL_PATH)

    # --- 4. Run prediction on the sample of images ---
    # Now, the 'source' is our list with the paths of the random images
    model.predict(
        source=amostra_de_imagens,
        save=True,
        imgsz=640,
        conf=0.5,
        project=projeto_path,
        name=f'{training_name}_img_test' # A more descriptive folder name
    )

    print("\nSample prediction completed!")
    print(f"The {len(amostra_de_imagens)} images with detections were saved in the folder: {projeto_path}/{training_name}_img_test")


In [ ]:
"""
This script prepares and exports a trained YOLO model to multiple formats 
(ONNX, OpenVINO, NCNN) and precision levels (FP32, FP16, INT8) optimized for 
edge deployment (e.g., on a Raspberry Pi 5). It copies the model locally for faster 
processing, exports it, renames the exported files/folders to prevent overwriting, 
compresses all outputs into a ZIP file, and copies the archive back to Google Drive.
"""
from ultralytics import YOLO
import os
import shutil

dataset_path = "dataset"
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2"
training_name = "modelo_V2_09_02_2026"

# --- Configurations ---
original_model_path = f'{projeto_path}/{training_name}/weights/best.pt'

# 1. LOCAL PREPARATION
local_export_dir = '/content/exportados_rpi5'
os.makedirs(local_export_dir, exist_ok=True)
local_model_path = f'{local_export_dir}/best.pt'

print("--- Moving model from Drive to local storage ---")
shutil.copy(original_model_path, local_model_path)
print("Model successfully copied to the local machine!")

imgsz = [640, 640]
data_yaml_path = f'{dataset_path}/data.yaml'

print(f"\nLoading model: {local_model_path}")
model = YOLO(local_model_path)

# Helper function to prevent files from being overwritten
def renomear_exportacao(caminho_antigo, novo_nome):
    novo_caminho = os.path.join(local_export_dir, novo_nome)
    # If it already exists from a previous run, delete it to avoid errors
    if os.path.exists(novo_caminho):
        if os.path.isdir(novo_caminho):
            shutil.rmtree(novo_caminho)
        else:
            os.remove(novo_caminho)
    os.rename(caminho_antigo, novo_caminho)
    print(f"-> Saved with the correct name: {novo_nome}")

# ============================================
# 1. Export to ONNX (FP32)
# ============================================
print("\n--- Exporting to ONNX (FP32) ---")
onnx_path = model.export(format='onnx', imgsz=imgsz, simplify=True, opset=12)
renomear_exportacao(onnx_path, 'modelo_onnx_fp32.onnx')

# ============================================
# 2. Export to ONNX (FP16)
# ============================================
print("\n--- Exporting to ONNX (FP16) ---")
onnx_fp16_path = model.export(format='onnx', imgsz=imgsz, simplify=True, half=True, opset=12)
renomear_exportacao(onnx_fp16_path, 'modelo_onnx_fp16.onnx')

# ============================================
# 3. Export to OpenVINO (INT8)
# ============================================
print("\n--- Exporting to OpenVINO (INT8) ---")
openvino_int8_path = model.export(format='openvino', imgsz=imgsz, int8=True, data=data_yaml_path)
renomear_exportacao(openvino_int8_path, 'int8_openvino_model')

# ============================================
# 4. Export to OpenVINO (FP16)
# ============================================
print("\n--- Exporting to OpenVINO (FP16) ---")
openvino_path = model.export(format='openvino', imgsz=imgsz, half=True)
renomear_exportacao(openvino_path, 'fp16_openvino_model')

# ============================================
# 5. Export to OpenVINO (FP32)
# ============================================
print("\n--- Exporting to OpenVINO (FP32) ---")
openvino_path = model.export(format='openvino', imgsz=imgsz)
renomear_exportacao(openvino_path, 'fp32_openvino_model')

# ============================================
# 6. Export to NCNN (FP32)
# ============================================
print("\n--- Exporting to NCNN (FP32) ---")
ncnn_fp32_path = model.export(format='ncnn', imgsz=imgsz)
renomear_exportacao(ncnn_fp32_path, 'ncnn_fp32')

# ============================================
# 7. Export to NCNN (FP16)
# ============================================
print("\n--- Exporting to NCNN (FP16) ---")
ncnn_fp16_path = model.export(format='ncnn', imgsz=imgsz, half=True)
renomear_exportacao(ncnn_fp16_path, 'ncnn_fp16')

# ============================================
# 8. Compress and Move to Drive
# ============================================
print("\n--- Compressing the exported models ---")
zip_base_name = '/content/modelos_exportados_rpi5'

# Compresses everything inside 'local_export_dir' (which now has the correct names!)
shutil.make_archive(zip_base_name, 'zip', local_export_dir)
print(f"ZIP file created locally at: {zip_base_name}.zip")

print("\n--- Moving ZIP file to Google Drive ---")
drive_dest_dir = f'{projeto_path}/{training_name}'
os.makedirs(drive_dest_dir, exist_ok=True)

drive_zip_path = f'{drive_dest_dir}/exportados_rpi5.zip'
shutil.copy(f'{zip_base_name}.zip', drive_zip_path)

print(f"\n✅ SUCCESS! Models generated with unique names and saved in Drive at: {drive_zip_path}")

In [ ]:
"""
This script evaluates multiple exported YOLO models (e.g., PyTorch, ONNX, OpenVINO) 
across different precision levels (FP32, FP16, INT8). It runs validation on the test 
dataset, extracts performance metrics (Precision, Recall, mAP) for each class, 
and compiles a comparative CSV report that is saved to Google Drive.
"""
import os
import shutil
import pandas as pd
from ultralytics import YOLO

dataset_path = "dataset"
local_export_dir = '/content/exportados_rpi5'

# ==========================================
# 2. Define models with the correct names
# ==========================================
modelos_para_testar = {
    "PyTorch_Original": f"{local_export_dir}/best.pt",
    "ONNX_FP32": f"{local_export_dir}/modelo_onnx_fp32.onnx",
    "ONNX_FP16": f"{local_export_dir}/modelo_onnx_fp16.onnx",
    "OpenVINO_FP32": f"{local_export_dir}/fp32_openvino_model",
    "OpenVINO_FP16": f"{local_export_dir}/fp16_openvino_model",
    "OpenVINO_INT8": f"{local_export_dir}/int8_openvino_model"
}

# ==========================================
# 3. Run tests and extract metrics
# ==========================================
resultados = []

for nome_modelo, caminho_modelo in modelos_para_testar.items():
    if not os.path.exists(caminho_modelo):
        print(f"⚠️ Skipping {nome_modelo}: File/Folder not found at {caminho_modelo}")
        continue

    print(f"\n========================================")
    print(f"🚀 Starting test: {nome_modelo}")
    print(f"========================================")

    try:
        model = YOLO(caminho_modelo)

        metrics = model.val(
            data=f'{dataset_path}/data.yaml',
            split='test',
            imgsz=640,
            plots=False,
            verbose=False
        )

        names = metrics.names
        map50_95 = metrics.box.maps
        precision = metrics.box.p
        recall = metrics.box.r
        map50_classes = metrics.box.all_ap[:, 0]
        map75_classes = metrics.box.all_ap[:, 5]

        for class_id, class_name in names.items():
            idx = int(class_id)
            resultados.append({
                "Model": nome_modelo,
                "Class": class_name,
                "Precision": round(precision[idx], 4),
                "Recall": round(recall[idx], 4),
                "mAP50": round(map50_classes[idx], 4),
                "mAP75": round(map75_classes[idx], 4),
                "mAP50-95": round(map50_95[idx], 4)
            })

        print(f"✅ {nome_modelo} successfully tested!")

    except Exception as e:
        print(f"❌ Error testing {nome_modelo}: {e}")

# TODO: Include inference time for CPU and GPU
# ==========================================
# 4. Generate and Save the CSV
# ==========================================
print("\n--- Compiling results into CSV ---")
df_resultados = pd.DataFrame(resultados)
print(df_resultados.head(12))

caminho_csv = '/content/comparativo_modelos.csv'
df_resultados.to_csv(caminho_csv, index=False)

# Copy to Drive so you don't lose it!
projeto_path = "/content/drive/MyDrive/Lesc-Bee videos/modelo_v2"
training_name = "modelo_V2_09_02_2026"
drive_csv_path = f'{projeto_path}/{training_name}/comparativo_modelos.csv'

# Ensure destination directory exists
os.makedirs(os.path.dirname(drive_csv_path), exist_ok=True)
shutil.copy(caminho_csv, drive_csv_path)

print(f"\n✅ All done! CSV saved to your Drive at: {drive_csv_path}")